# Otimização de preço no e-commerce brasileiro
## Roteiro por níveis

**Repositório:** https://github.com/BernardoliveiraFiap/price-optimization-olist
**Página do projeto:** https://bernardoliveirafiap.github.io/price-optimization-olist/

### O único conceito que você precisa antes de começar

**Elasticidade** é quanto a quantidade vendida muda quando o preço muda 1%. É um número
negativo, porque preço sobe e venda cai.

Ela é uma **taxa**, não um valor de venda: `-2` quer dizer "para cada 1% que o preço sobe, a
quantidade vendida cai 2%". Vale nos dois sentidos, então `-2` também quer dizer que baixar 1%
o preço faz a venda subir 2%. Sozinha ela não prevê nada, só responde depois que você diz
quanto mexeu no preço.

Onde ela cai muda a decisão:

* entre `0` e `-1`: demanda **inelástica**, a venda cai menos do que o preço sobe, então
  aumentar preço aumenta a receita.
* abaixo de `-1`: demanda **elástica**, a venda cai mais do que o preço sobe, então aumentar
  preço destrói receita.

O `-1` é a fronteira. Todo número que aparecer neste notebook se lê contra ela.

### Os níveis

| Nível | O que você vai fazer |
|---|---|
| 1 | Preparar o ambiente |
| 2 | Trazer o dataset do Olist |
| 3 | Olhar o dado cru antes de modelar |
| 4 | Testar os estimadores onde a resposta certa é conhecida |
| 5 | Montar o painel produto por semana |
| 6 | Rodar o pipeline completo no dado real |
| 7 | Rodar o mesmo pipeline no mercado sintético |
| 8 | Prever demanda com machine learning |
| 9 | Desenhar os gráficos a partir dos CSVs de saída |
| 10 | Rodar a suíte de testes |

## Nível 1. Preparar o ambiente

Baixa o código do GitHub e instala a única biblioteca que o Colab não traz de fábrica: o
`pulp`, que é o solver do problema de otimização. Pandas, statsmodels, scikit-learn, pyarrow e
PyTorch já vêm instalados aqui.

O repositório não é um pacote publicado no PyPI, então a célula também aponta o `PYTHONPATH`
para a pasta `src`. Guarde essa linha: é ela que faz `python -m pricing.alguma_coisa` funcionar
nos próximos níveis.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/price-optimization-olist")
URL = "https://github.com/BernardoliveiraFiap/price-optimization-olist.git"

if not (REPO / "src").exists():
    # clona num diretorio limpo e copia por cima, para o caso de ja existir
    # alguma coisa em REPO (dados soltos, por exemplo)
    tmp = Path("/content/_clone_tmp")
    shutil.rmtree(tmp, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", URL, str(tmp)], check=True)
    shutil.copytree(tmp, REPO, dirs_exist_ok=True)
    shutil.rmtree(tmp, ignore_errors=True)
    print("codigo baixado do GitHub")
else:
    print("codigo ja estava aqui, clone pulado")

os.chdir(REPO)
os.environ["PYTHONPATH"] = str(REPO / "src")   # o repositorio nao e um pacote instalado
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pulp>=2.8,<4"], check=True)

from importlib.metadata import version

print("\nPython        " + sys.version.split()[0])
for nome in ["numpy", "pandas", "scipy", "statsmodels", "scikit-learn", "pulp", "pyarrow", "torch"]:
    try:
        print(f"{nome:<14}{version(nome)}")
    except Exception:
        print(f"{nome:<14}nao instalado")

import pricing

ultimo = subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print("\nmodulo pricing importado de:", Path(pricing.__file__).parent)
print("ultimo commit:", ultimo)
print("pasta atual:", os.getcwd())

Se apareceu a lista de versões e o caminho do módulo `pricing`, o ambiente está pronto.

**Explore antes de seguir.** Abra o painel de arquivos à esquerda, o ícone de pasta, e navegue
até `price-optimization-olist`. Dê uma olhada no `Makefile` e no `README.md`. Você vai voltar
neles em quase todos os níveis.

## Nível 2. Trazer o dataset do Olist

O dataset é público, são cerca de 100 mil pedidos reais de uma marketplace brasileira entre
2016 e 2018, mas o download direto exige login na Kaggle. Por isso os CSVs vêm de arquivo
zipado, e não de download.

A célula procura qualquer `.zip` em `/content`, extrai os CSVs de todos eles para `data/raw/`
e segue. Se não achar zip nenhum, aí sim ela abre o botão **Escolher arquivos** para você
mandar o seu. Rodar de novo é seguro: ela detecta o que já está no disco e pula tudo.

Para colocar um zip em `/content` sem passar pelo botão, arraste o arquivo para o painel de
arquivos à esquerda, o ícone de pasta.

Fonte original, se um dia precisar baixar de novo:
https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

In [ ]:
import zipfile
from pathlib import Path

RAW = Path("/content/price-optimization-olist/data/raw")
RAW.mkdir(parents=True, exist_ok=True)

NECESSARIOS = [
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_products_dataset.csv",
    "olist_customers_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "product_category_name_translation.csv",
]


def faltando():
    return [n for n in NECESSARIOS if not (RAW / n).exists()]


def extrair(caminho_zip):
    with zipfile.ZipFile(caminho_zip) as z:
        for membro in z.namelist():
            nome = Path(membro).name
            if nome.lower().endswith(".csv"):
                (RAW / nome).write_bytes(z.read(membro))
    print(f"extraido: {caminho_zip.name}")


if faltando():
    zips = sorted(Path("/content").glob("*.zip"))

    if not zips:
        from google.colab import files

        print("Selecione o zip com os CSVs (esta na sua Area de Trabalho).\n")
        enviados = files.upload()
        for nome in enviados:
            origem = Path(nome).resolve()
            destino = Path("/content") / origem.name
            if origem != destino:
                origem.replace(destino)
            if destino.suffix.lower() == ".zip":
                zips.append(destino)
            else:
                destino.replace(RAW / destino.name)

    for caminho in zips:
        extrair(caminho)
else:
    print("Os CSVs ja estao em data/raw. Upload pulado.")

restante = faltando()
if restante:
    raise SystemExit("Ainda faltam arquivos: " + ", ".join(restante))

for p in sorted(RAW.glob("*.csv")):
    print(f"{p.stat().st_size / 1024 / 1024:8.2f} MB  {p.name}")

Os 6 arquivos que o projeto usa estão em `data/raw/`. O de geolocalização, que tem 61 MB, ficou
de fora do zip porque o modelo não usa.

A partir daqui cada nível tem o comando pronto, as perguntas que ele deveria te fazer, e
as respostas logo abaixo delas. Tente responder antes de ler.

## Nível 3. Olhar o dado cru antes de modelar

**O que este nível faz.** Abre dois dos CSVs com pandas e imprime o tamanho da base, o
período que ela cobre, as 5 primeiras linhas da tabela de itens e a contagem de status dos
pedidos. Nada de modelo ainda, só olhar o que tem dentro do arquivo.

**Onde isso vira código de verdade.** A função `load_transactions`, em
`src/pricing/data.py`, é quem junta esses arquivos no projeto. Vale abrir depois de rodar.

### Perguntas e respostas

1. Quantos itens vendidos existem, e quantos produtos distintos?

   **112.650 itens**, **32.951 produtos** e 99.441 pedidos. Divida os dois primeiros: dá
   menos de 4 vendas por produto na vida inteira do catálogo. Segure esse número, é ele que
   vai derrubar 97% do catálogo no nível 5.

2. Por que o número de itens não bate com o de pedidos?

   Porque as duas tabelas contam coisas diferentes. Uma compra com um teclado e dois mouses
   é **1 linha** em pedidos e **3 linhas** em itens. A saída mostra 1,13 item por compra: a
   maioria das compras tem um produto só, por isso os dois números ficam parecidos sem serem
   a mesma coisa.

3. Que período o dataset cobre?

   De **04/09/2016 a 17/10/2018**, 110 semanas.

4. Olhando as 5 primeiras linhas, por que o preço está na tabela de itens e não na de pedidos?

   Porque cada produto tem o seu preço. Naquela compra de teclado e dois mouses existem 3
   preços, e um campo único no pedido teria que escolher qual guardar. Repare também que
   `order_item_id` é só o contador do item **dentro** daquele pedido, por isso quase toda
   linha traz `1`. Ele não identifica nada sozinho.

5. O que é o `freight_value` e por que ele não pode entrar como preço?

   É o frete daquele item. Quem paga é o comprador, mas o dinheiro vai para a transportadora,
   não para o vendedor. Somar ele ao preço colocaria distância e peso dentro do valor do
   produto, e o modelo acabaria medindo o CEP do cliente junto com a decisão de comprar.

6. Quantos pedidos não foram entregues, e por que isso importa?

   **2.963**, ou 3,0% dos 99.441. A maioria é `shipped` (1.107) e `canceled` (625). Eles saem
   da conta no nível 5, porque um pedido cancelado tem preço registrado mas nunca virou venda.
   Contar ele seria registrar procura que não existiu.

7. Qual coluna você esperaria encontrar e não está em nenhum dos 7 arquivos?

   **Custo.** Sem custo não se calcula margem, então lá no nível 6 o projeto vai ter que
   assumir um valor, 35%. É a maior limitação do trabalho, e ela já estava visível aqui.

In [ ]:
from pathlib import Path

import pandas as pd

# Depois do nivel 1 o diretorio de trabalho ja e a raiz do repositorio, entao
# "data/raw" tambem funcionaria. Uso o caminho absoluto para a celula rodar
# mesmo se voce tiver mudado de pasta no meio do caminho.
RAW = Path("/content/price-optimization-olist/data/raw")

# As duas tabelas deste nivel tem GRANULARIDADE diferente, e isso e o ponto:
# "itens" tem uma linha por ITEM vendido, "pedidos" tem uma linha por PEDIDO.
# Uma compra com 3 produtos aparece 1 vez em pedidos e 3 vezes em itens.
itens = pd.read_csv(RAW / "olist_order_items_dataset.csv")
pedidos = pd.read_csv(RAW / "olist_orders_dataset.csv")

# --- 1. Tamanho da base -------------------------------------------------
print(f"itens vendidos      {len(itens):,}")
print(f"produtos distintos  {itens['product_id'].nunique():,}")
print(f"pedidos             {len(pedidos):,}")

# A razao entre os dois deixa a diferenca de granularidade explicita.
print(f"itens por pedido    {len(itens) / len(pedidos):.2f}")

# --- 2. Periodo coberto -------------------------------------------------
# A coluna chega como TEXTO. Sem converter para data, min e max comparariam
# as strings em ordem alfabetica em vez de cronologica.
pedidos["order_purchase_timestamp"] = pd.to_datetime(pedidos["order_purchase_timestamp"])

inicio = pedidos["order_purchase_timestamp"].min()
fim = pedidos["order_purchase_timestamp"].max()
print(f"\nperiodo             {inicio:%d/%m/%Y} ate {fim:%d/%m/%Y}")
print(f"semanas no total    {(fim - inicio).days // 7}")

# --- 3. A cara de uma linha --------------------------------------------
# Repare em quais colunas existem aqui, e principalmente em qual NAO existe
# em lugar nenhum do dataset.
print("\ncolunas de itens:", list(itens.columns))
display(itens.head())

# --- 4. Status dos pedidos ---------------------------------------------
# O projeto so usa os pedidos com status "delivered".
print("\npedidos por status")
print(pedidos["order_status"].value_counts())

nao_entregues = (pedidos["order_status"] != "delivered").sum()
print(f"\nnao entregues       {nao_entregues:,} ({nao_entregues / len(pedidos):.1%} do total)")

## Nível 4. O simulado com gabarito

**O que este nível faz.** O código inventa 20 mercados falsos usando uma elasticidade que ele
mesmo escolheu. Depois solta três estimadores em cima desses mercados e mede o quanto cada um
chegou perto do número que ele escolheu. Nenhum CSV é lido aqui: os dados nascem na hora.

Ele existe porque no Olist ninguém sabe a resposta certa. É como testar uma balança pondo em
cima um peso de 1 kg que você já sabe que é 1 kg: se ela marcar 800 g, você descobre o tamanho
e a direção da mentira dela.

**Onde isso vira código de verdade.** `src/pricing/evaluate/validation.py` monta os mercados,
e o `Makefile` tem o comando no alvo `validate`.

### Perguntas e respostas

1. A saída compara os estimadores contra o quê?

   Contra a primeira coluna, `true_beta` = **-2,103**. É o gabarito, o número que o código
   escolheu antes de gerar as vendas.

   (Lê-se **menos dois vírgula cento e três**, não dois mil e cento e três. É uma
   elasticidade, ou seja: **para cada 1% que o preço sobe, a quantidade vendida cai 2,103%**,
   e para cada 1% que ele cai, a venda sobe 2,103%. É uma taxa, como "km por litro": sozinha
   não diz quanto caiu, só responde depois que você diz quanto mexeu no preço. Num produto de
   R\$ 50 vendendo 25 unidades por semana, subir para R\$ 55, que é 10% a mais, derruba a venda
   para 20,5 unidades.)

   Uma sutileza que confunde depois: cada um dos 20 mercados tem o **seu próprio** gabarito,
   sorteado entre `-2,8` e `-1,3`. O `-2,103` da tabela é a **média** dos 20, que vão de -2,41
   a -1,77. É por isso que o nível 7, que roda um mercado só, mostra um gabarito diferente,
   `-2,319`, sem que nenhum dos dois esteja errado.

2. O que é a coluna `bias`?

   É uma subtração: `mean_estimate - true_beta`. No OLS dá `-1,590 - (-2,103) = 0,513`. É a
   distância entre a resposta média do estimador e o gabarito. A coluna do lado,
   `rel_bias_pct`, é a mesma coisa em porcentagem: `0,513 / 2,103` dá os 24,5%.

   Traduzindo o `0,513` para português: o OLS acredita que 1% de aumento no preço custa 1,59%
   da venda, quando custa 2,10%. Ele enxerga o cliente mais tolerante a preço do que ele é.

   E isso vira dinheiro na fórmula do preço ótimo, `p* = c x beta / (1 + beta)`. Um produto de
   custo R\$ 26,22: com o gabarito `-2,103` a fórmula manda cobrar **R\$ 50**; com o `-1,590` do
   OLS, a mesma fórmula manda cobrar **R\$ 70,66**, 41% a mais. O `0,513` da coluna `bias` é
   esse erro de 41% no preço, escrito na unidade da elasticidade.

3. Por que rodar 20 mercados e não um só?

   Porque em um mercado só o estimador pode ter tido sorte. Viés é a distância da **média**
   das 20 até o gabarito, e ele é o erro que não some repetindo. Uma balança descalibrada que
   marca sempre 200 g a mais não melhora se você pesar mil vezes.

4. O que a coluna `ci_coverage` mede, e por que `0,000` é o número mais grave da tabela?

   Todo estimador entrega um intervalo do tipo "a resposta está entre X e Y, com 95% de
   confiança". Essa coluna conta em quantas das 20 rodadas o intervalo realmente continha o
   gabarito. Deveria dar perto de 0,95. O OLS deu **zero**: em 20 tentativas, nenhuma. Ele não
   só erra, ele erra dizendo que tem certeza, e por isso ninguém desconfia dele.

5. Os três `bias` são positivos. Por que essa direção é a cara?

   Positivo aqui quer dizer estimativa **mais perto do zero** que a verdade, ou seja, a
   demanda parece menos sensível a preço do que é. Traduzindo para a decisão: subir o preço
   parece mais seguro do que é. De todos os erros possíveis, é o que custa dinheiro.

6. O que o 2SLS usa que os outros dois não usam?

   Um **instrumento**. É alguma coisa que empurra o preço por um motivo que nada tem a ver com
   a vontade de comprar, e aqui é o custo de produção. Ele fica só com a parte do preço que se
   mexeu por causa desse empurrão e joga fora o resto, que é justamente a parte contaminada
   pela qualidade do produto. Os outros dois olham o preço inteiro, sujeira incluída.

7. E a segunda célula, com `--exogenous`, o que ela mostra?

   Ela repete tudo com a endogeneidade **desligada**, ou seja, num mundo onde o preço não
   reage à qualidade. Ali os efeitos fixos ficam praticamente sem viés (+0,02%) e o OLS
   continua errado (+19,7%). Cada estimador falha exatamente no caso que as premissas dele
   proíbem, e isso é o que separa um teste bom de uma tabela bonita.

In [ ]:
# O Makefile tem "$(PY) -m pricing.evaluate.validation --reps 20". No Colab
# nao existe .venv, entao o $(PY) vira python.
#
#   !                     nao e Python, e o Jupyter mandando a linha pro terminal
#   python -u             desliga o buffer, a saida aparece enquanto roda
#   -m pricing.evaluate.validation   roda o MODULO como programa. So funciona
#                         porque o nivel 1 colocou src no PYTHONPATH
#   --reps 20             20 mercados sinteticos independentes. Um so seria sorte
!python -u -m pricing.evaluate.validation --reps 20

In [ ]:
# O experimento de controle: mesma coisa com a endogeneidade DESLIGADA.
# Compare as duas tabelas linha a linha.
!python -u -m pricing.evaluate.validation --reps 20 --exogenous

## Nível 5. Montar o painel produto por semana

**O que este nível faz.** Pega a lista de itens vendidos e transforma numa tabela com **uma
linha por produto por semana**, contendo preço, unidades e o instrumento. Grava o resultado em
`data/processed/panel.parquet`. É o único nível que escreve um arquivo que os próximos leem.

A segunda célula compara a receita dos produtos que sobreviveram contra a receita total.

**Onde isso vira código de verdade.** `src/pricing/data.py`, alvo `panel` do `Makefile`.

### Perguntas e respostas

1. Quantos produtos entraram e quantos sobraram?

   Entraram **32.216** e sobraram **844**. Repare que 32.216 é menor que os 32.951 do nível 3:
   a diferença são os produtos que só apareceram em pedidos não entregues, e que já foram
   descartados aqui.

2. Por que exigir 12 semanas de histórico?

   Porque a elasticidade se mede olhando o preço **mexer** e a venda reagir. Um produto que
   vendeu 3 vezes em dois anos, sempre pelo mesmo preço, não tem nada para medir. Ele não daria
   uma estimativa ruim, daria uma estimativa inventada.

3. Sobraram 2,6% do catálogo. Então o modelo só serve para 2,6% da loja?

   Não, e a segunda célula mostra por quê. Esses 844 produtos são **R\$ 3,37 milhões de
   R\$ 13,59 milhões**, ou seja **24,8% da receita**. Poucos produtos, um quarto do dinheiro. É
   essa a resposta quando alguém perguntar se o trabalho serve para a loja inteira.

4. O que é o "instrumento" que essa célula calcula?

   É o preço médio da **mesma categoria, na mesma semana, em outros estados**. Serve para o
   nível 6: para saber se o preço causa a venda, você precisa de algo que empurre o preço por
   um motivo alheio à vontade de comprar. Custo seria perfeito, mas o Olist não tem essa coluna.
   A aposta é que custo e frete sobem juntos em regiões diferentes, enquanto a procura é local.
   Isso é uma aposta, e o nível 6 testa se ela se sustenta.

5. Por que o painel tem 88 semanas se o dataset cobre 110?

   Porque as 22 que faltam são semanas em que nenhum dos 844 produtos sobreviventes teve venda,
   nas duas pontas do período.

In [ ]:
# Alvo "panel" do Makefile. Le os 6 CSVs, junta tudo, agrupa por produto e
# semana, calcula o instrumento, corta quem tem historico curto demais e
# grava data/processed/panel.parquet. E o unico nivel que ESCREVE um arquivo
# que os proximos vao ler.
#
#   --level product    uma linha por produto por semana (padrao do projeto)
#   --level category   uma linha por categoria por semana, menos detalhe
#   --min-weeks 12     semanas de historico exigidas para o produto entrar
!python -u -m pricing.data --level product

In [ ]:
import pandas as pd

# Quantos produtos sobreviveram voce ja viu acima. Esta celula mede outra
# coisa: quanto de DINHEIRO esses sobreviventes representam.
itens = pd.read_csv("data/raw/olist_order_items_dataset.csv")
painel = pd.read_parquet("data/processed/panel.parquet")

receita_total = itens["price"].sum()
receita_painel = painel["revenue"].sum()

print(f"produtos no catalogo   {itens['product_id'].nunique():,}")
print(f"produtos no painel     {painel['unit_id'].nunique():,}")
print(f"\nreceita total          R$ {receita_total:,.0f}")
print(f"receita do painel      R$ {receita_painel:,.0f}")
print(f"fatia da receita       {receita_painel / receita_total:.1%}")

## Nível 6. O pipeline completo no dado real

**O que este nível faz.** Lê o painel do nível 5 e vai até o fim: estima a elasticidade com os
três estimadores, testa se o instrumento presta, escolhe em quem confiar, estima categoria por
categoria, descarta as categorias que não dá para precificar e roda o otimizador, que escolhe
um preço para cada produto respeitando os limites da empresa.

Leva alguns minutos e a saída é longa.

### Antes de ler a saída, duas coisas

**1. O ponto é a vírgula decimal.** O programa imprime em inglês:

| Está escrito na tela | Se lê |
|---|---|
| `58.440` | 58,44 |
| `147.697` | 147,7 |
| `-0.756` | -0,756 |

Nenhum número dessa saída está na casa dos milhares.

**2. O que a elasticidade quer dizer, com número na mão.** Ela é quantos por cento a quantidade
vendida muda a cada 1% de mudança no preço. Pegue um produto que vende **100 unidades por
semana a R\$ 100** e suba o preço para R\$ 101, que é 1% a mais:

| Elasticidade | Quantas unidades o modelo prevê |
|---|---|
| `-0,756` | 99,3. Alguns clientes desistem. |
| `-0,002` | 100,0. Ninguém desiste. |
| `+58,44` | **178,9**. Vende quase o dobro por estar mais caro. |

E se você subir 10%, para R\$ 110: o de `-0,756` prevê 93 unidades, o de `-0,002` prevê 100, e o
de `+58,44` prevê **26.241 unidades**. Esse último é o número que o pipeline vai jogar fora, e o
resto deste nível é a explicação de como ele descobre isso sozinho.

### Perguntas e respostas

1. O primeiro bloco imprime três números para a carteira inteira. O que eles são?

   Três **elasticidades**, ou seja, três tentativas de medir a mesma coisa: OLS `-0,002`, efeitos
   fixos `-0,756` e 2SLS `+58,44`. Não são erros nem porcentagens de nada, são as três respostas
   que cada método dá para "quanto a venda cai quando o preço sobe 1%". Elas discordam porque
   cada uma assume uma coisa diferente sobre de onde vieram os preços do Olist.

2. O que é o `SE` que aparece ao lado de cada um?

   É o **erro padrão**, o chacoalho da estimativa: o quanto ela mudaria se você repetisse a
   medição com outro pedaço de dado.

   Efeitos fixos deram `-0,756` com erro `0,092`. Isso é uma medição: a resposta está por volta
   de -0,76, com uma margem estreita para cada lado.

   O 2SLS deu `+58,44` com erro `147,7`. Isso não é medição, é chute: a margem é maior que o
   próprio número, e a resposta poderia estar em qualquer lugar entre -237 e +353.

3. Por que o pipeline descartou o 2SLS?

   Pelo **F do primeiro estágio**, que deu `0,16`. O mínimo aceito é 10.

4. O que é esse F, em português?

   O 2SLS trabalha em dois passos. No primeiro ele tenta explicar o preço usando só o instrumento
   (o preço da mesma categoria, mesma semana, em outros estados). O F mede se esse primeiro passo
   funcionou: se quando o instrumento se mexe, o preço se mexe junto.

   `0,16` quer dizer que não se mexe quase nada. O segundo passo então acaba dividindo por um
   número perto de zero, e é exatamente daí que sai o `+58,44` com erro de `147,7`.

5. Por que o instrumento funcionou no nível 4 e morreu aqui?

   Porque **só 1,1% da variação do preço acontece dentro do mesmo produto**. Um anúncio do Olist
   nasce com um preço e morre com ele. Os outros 98,9% da variação são entre produtos diferentes,
   e essa parte a conta de efeitos fixos remove de propósito, porque é ela que carrega a
   qualidade que ninguém observa. Sobra 1,1% para o instrumento trabalhar, e sobre 1,1% não dá
   para concluir nada.

6. O que o pipeline fez ao descobrir isso?

   Recusou o próprio 2SLS e ficou com os efeitos fixos: **-0,756**. Traduzindo para o produto de
   100 unidades: subir 1% o preço custa 0,7 unidade de venda.

   Vale reparar no comportamento, não só no número. O caminho comum seria publicar o `+58,44`
   ou esconder o teste que reprovou.

7. Depois disso ele estima categoria por categoria. Quantas existem e quantas passaram?

   O painel tem **20 categorias** e passaram **3**: `cool_stuff` (-1,44), `watches_gifts` (-1,22)
   e `perfumery` (-1,20).

   Traduzindo o `-1,44`: em `cool_stuff`, cada 1% de aumento no preço derruba 1,44% da venda.
   Como 1,44 é maior que 1, a venda cai mais do que o preço sobe — e é justamente isso que faz
   existir um preço ótimo com valor finito. Nas outras 17, onde a queda é menor que o aumento,
   a conta manda subir o preço para sempre, que é o assunto da pergunta 9.

8. Passaram em quê? Qual é o filtro?

   Duas condições ao mesmo tempo, escritas em `src/pricing/pipeline.py`:

   - a elasticidade tem que ser menor que **-1,05**, ou seja, elástica de verdade e não raspando
     o -1;
   - o `|t|` tem que ser maior que **2**. O `t` é a estimativa dividida pelo próprio erro padrão,
     então ele mede se o número é grande comparado ao chacoalho dele. Abaixo de 2, a estimativa
     pode ser só sorte.

   As duas categorias que deram número positivo têm `|t|` menor que 1: o chacoalho é maior que a
   própria estimativa. Não são produtos que vendem mais quando encarecem, são contas sem
   informação suficiente.

9. Por que jogar fora 17 categorias em vez de precificar elas também?

   Porque a elasticidade delas caiu entre `0` e `-1`. Nessa faixa, cada 1% de aumento no preço
   custa menos de 1% de venda, então subir o preço aumenta a receita **sempre**. A conta de preço
   ótimo manda subir até o infinito, e a única coisa que segura é o teto que você deu de presente
   ao modelo. A "recomendação" seria uma conclusão sobre o teto, não sobre o mercado.

10. O que são as "travas de negócio" que aparecem no otimizador?

    São limites que a empresa impõe para o otimizador não fazer besteira. Estão em
    `src/pricing/config.py`:

    | Trava | Valor | O que impede |
    |---|---|---|
    | Banda por produto | ±30% | nenhum produto se afasta mais que isso do preço de hoje |
    | Teto de preço médio | +5% | a carteira toda não pode encarecer mais que isso na média |
    | Piso de receita | -2% | a operação não pode perder mais faturamento que isso |
    | Piso de volume | desligado | opcional, é ele que os cenários "crescer volume 10%" ligam |

    O otimizador também não testa infinitos preços: testa **11 preços por produto**, espalhados
    dentro da banda de ±30%.

11. Quanto ele entregou com as travas ligadas?

    Ele trabalha sobre **75 produtos**, os das 3 categorias aprovadas, que são 23% da receita do
    painel. Resultado: margem **+7,56%**, receita `-1,04%`, volume `-6,01%` e preço médio
    `+5,00%`, que é exatamente o teto da política. Ele encostou no limite.

    (Cuidado com o `+7,56%`: é o quanto a margem **cresce**, não a margem em si. Se hoje sobram
    R\$ 100 de margem, passam a sobrar R\$ 107,56. E repare que receita e volume **caem**: o
    ganho vem de vender um pouco menos, mais caro, com mais dinheiro sobrando no fim.)

    Sobre "margem": como o Olist não registra custo, o projeto assume margem bruta de **35%** da
    receita. Esse 35% é premissa, não dado, e é a maior fraqueza do trabalho.

12. E sem trava nenhuma?

    **+33,34%** de margem. Só que para isso ele sobe 30% o preço médio, que é o limite da banda
    por produto, e perde **28% do volume**. Numa empresa de verdade isso não é recomendação.

13. Então quanto custam as travas?

    `33,34 - 7,56 = ` **25,79 pontos percentuais** de margem. Guarde esse número: no nível 7,
    no mercado inventado, essa mesma conta vai dar zero.

    ("Pontos percentuais" porque é a subtração direta de duas porcentagens, não uma porcentagem
    de outra. É o tanto de ganho que a empresa abre mão para não fazer nada arriscado com o
    preço.)

In [ ]:
# Alvo "run-olist" do Makefile. Este e o nivel principal e leva alguns
# minutos. Ele faz, nesta ordem: estima a elasticidade da carteira com os
# tres metodos, testa o instrumento e escolhe em quem confiar, estima
# categoria por categoria, descarta as que nao podem ser precificadas e roda
# o otimizador com as travas de negocio.
#
# A saida e longa. Leia de cima para baixo, ela sai em blocos.
!python -u -m pricing.pipeline --source olist

## Nível 7. O mesmo pipeline no mercado inventado

**O que este nível faz.** Roda exatamente o mesmo código do nível 6, trocando só a fonte dos
dados: em vez do Olist, o mercado gerado pelo próprio projeto, o mesmo do nível 4. Aqui a
elasticidade verdadeira existe e o instrumento de custo é de verdade.

Serve para você ver o pipeline funcionando no melhor caso possível e comparar com o que
aconteceu no dado real. Vale abrir os dois lado a lado.

### Perguntas e respostas

1. Qual o tamanho do painel aqui?

   **31.200 linhas, 300 produtos, 104 semanas**. No Olist eram 16.278 linhas e 844 produtos. Este
   é maior porque foi construído para ser: cada produto inventado vende toda semana.

2. Quais elasticidades saíram, e qual é a verdadeira?

   OLS `-1,776`, efeitos fixos `-2,153`, 2SLS `-2,319`. E a linha `TRUE` diz `-2,319`, que é o
   gabarito que o gerador usou. O 2SLS cravou.

   (`-2,319` quer dizer: cada 1% de aumento no preço derruba 2,319% da venda. E não, ele não
   briga com o `-2,103` do nível 4: lá eram 20 mercados e a tabela mostrava a **média** dos 20
   gabaritos, aqui é **um mercado só**, com o gabarito dele.)

   Medindo cada um contra esse gabarito: o 2SLS acertou até a terceira casa, os efeitos fixos
   erraram 0,17 (7%) e o OLS errou 0,54 (23%). Os dois que erraram foram para o mesmo lado, o
   de "o cliente aguenta mais aumento do que aguenta de verdade".

3. O que o F do primeiro estágio deu?

   Cerca de **122 mil**, contra `0,16` do nível 6. Aqui o instrumento explica o preço com folga,
   então o pipeline aceitou o 2SLS e usou ele para decidir.

4. Quantas categorias passaram no filtro?

   **6 de 6**, cobrindo 100% da receita do painel. No Olist foram 3 de 20, cobrindo 23%.

5. Quanto custaram as travas de negócio?

   **0,00 pontos**. Com trava e sem trava dá o mesmo `+6,89%` de margem, porque o preço ótimo de
   cada produto já cabia dentro da política sozinho. Ninguém precisou ser segurado.

6. Se a margem final é parecida nos dois (`+6,89%` aqui, `+7,56%` lá), o que mudou de verdade?

   Mudou de onde o ganho nasce.

   Aqui ele nasce da **estimativa**: o modelo sabe a elasticidade de cada categoria, o otimizador
   só executa, e as travas não atrapalham.

   No dado real a estimativa é fraca, 17 das 20 categorias foram descartadas, e o que segura o
   resultado são as travas. Tanto que elas custam 25,79 pontos lá e zero aqui.

   É por isso que o projeto roda os dois. Sem essa comparação, o `+7,56%` do nível 6 pareceria um
   sucesso simples, quando na verdade é um resultado que depende quase todo da política.

In [ ]:
# Mesmo comando, so muda a fonte: agora o mercado inventado do nivel 4, onde
# a elasticidade verdadeira existe e o instrumento de custo e de verdade.
# E o contrafactual do nivel 6.
!python -u -m pricing.pipeline --source synth

## Nível 8. Machine learning: prever quantas unidades o produto vende

**O que este nível faz.** Tenta adivinhar **quantas unidades um produto vai vender na semana**,
que é a coluna `log_units` do painel. Sem causa, sem elasticidade, só acertar o número.

Para adivinhar, ele olha **9 informações** de cada linha:

| Grupo | O que é |
|---|---|
| Do produto naquela semana | preço, frete, nota média das avaliações, quantos vendedores estavam ativos na categoria |
| Identificação | a categoria e o número da semana |
| Histórico do próprio produto | quanto vendeu 1, 2 e 4 semanas atrás, a média das 4 últimas semanas e o preço da semana passada |

**Onde isso vira código de verdade.** `src/pricing/demand/ml_forecast.py`.

### Perguntas e respostas

1. O que é "um modelo" aqui?

   Qualquer regra que pega essas 9 informações e devolve um número. Só isso.

   Segure uma linha na mão para acompanhar: produto A, semana 30, preço R\$ 89, frete R\$ 15, nota
   4,2, 12 vendedores na categoria, vendeu 5 na semana 29, 3 na 28 e 6 na 26. A pergunta é
   quantas unidades ele vende na semana 30.

2. Por que a saída traz três modelos e não um?

   Porque um erro sozinho não quer dizer nada. Se eu te falo que o erro do modelo é `0,4489`,
   isso é bom ou ruim? Só dá para saber com algo do lado para comparar.

3. O que cada um dos três faz com aquela linha?

   **O burro de propósito**, chamado de `persistence` na saída: chuta que vende o mesmo que
   vendeu semana passada, ou seja **5**. Ignora as outras 8 informações. Está ali de régua.

   **A fórmula fixa**, chamada de `ridge`: monta uma conta com um peso para cada informação, tipo
   `unidades = (peso_1 x preço) + (peso_2 x frete) + ...`. Treinar é descobrir esses pesos
   olhando o passado. Uma vez descobertos, valem para tudo: o mesmo peso do preço se aplica a
   perfumaria em dezembro e a relógio em julho.

   **As perguntas encadeadas**, chamado de `gbm`: não monta fórmula. Faz perguntas de sim ou não,
   tipo "vendeu mais de 4 na semana passada? sim. O preço está acima de 80? sim. Tem mais de 10
   vendedores na categoria? não. Então chuto 6". Isso é uma árvore. Como uma árvore sozinha erra
   bastante, ele constrói **400 em fila**: a segunda aprende só o erro que a primeira deixou, a
   terceira o que sobrou da segunda, e o chute final é a soma de todas.

4. Por que as perguntas encadeadas ganham da fórmula fixa?

   Porque a fórmula tem um peso de preço só, para o dataset inteiro. A árvore consegue dizer
   "para perfumaria em dezembro o preço pesa muito, para relógio em julho não pesa", já que cada
   caminho de perguntas leva a uma resposta diferente.

5. O que é o RMSE da tabela?

   É a distância média entre o que o modelo chutou e o que aconteceu de verdade, nas semanas que
   ele nunca viu. Menor é melhor. O valor absoluto não tem tradução direta em unidades, porque a
   conta é feita no logaritmo da quantidade; o que vale é comparar os três entre si.

6. Qual ganhou?

   | Regra | RMSE |
   |---|---|
   | O burro | 0,7155 |
   | A fórmula fixa | 0,5739 |
   | As perguntas encadeadas | **0,4489** |

   As perguntas encadeadas erram 37% menos que o burro.

7. Por que a validação treina só no passado?

   Porque o jeito normal, que é sortear linhas ao acaso, aqui seria trapaça: o modelo treinaria
   com a semana 40 de um produto para adivinhar a semana 12 do mesmo produto, ou seja, veria o
   futuro. Então o código treina até certa semana e testa nas seguintes, empurrando essa
   fronteira 4 vezes.

8. De onde sai a linha `implied elasticity`, se o modelo nunca fala de elasticidade?

   O código cutuca o modelo. Pega o bloco de teste, congela tudo (mesmo produto, mesmo histórico,
   mesma semana), empurra **só o preço em 5%** e pede a previsão de novo. O quanto a previsão se
   mexeu, dividido pelos 5%, é a elasticidade que aquele modelo implicaria se alguém usasse ele
   para definir preço.

9. Deu quanto, e como se compara com o nível 6?

   Deu **-0,281**, contra `-0,756` do modelo causal. No produto de 100 unidades a R\$ 100 isso é a
   diferença entre prever que você perde 0,3 unidade ao subir 1% e prever que perde 0,7. Um erro
   de mais de duas vezes, sempre para o lado de "pode subir o preço".

   E o código roda tudo de novo **sem o histórico de vendas**, para ninguém dizer que a culpa é
   das defasagens: aí vai para **-0,050**, quase o `-0,002` do OLS ingênuo do nível 6.

   `-0,050` quer dizer, na prática, "o preço não afeta a venda": subir 1% o preço custaria
   0,05% do volume, meia unidade a cada mil. Nenhum mercado se comporta assim. O modelo não
   mediu o efeito do preço fraco demais, ele apagou o efeito do preço.

10. O que se conclui disso?

    Que o melhor previsor é o pior modelo de preço. Prever o que vai acontecer sem mexer em nada
    e adivinhar o que aconteceria **se** você mexesse são perguntas diferentes. Trocar de
    algoritmo não conserta viés, porque o viés está no dado e não no estimador.

In [ ]:
# Um modelo de arvores com gradient boosting, treinado para PREVER demanda e
# nao para estimar causa. --folds 4 sao 4 divisoes de validacao em janela
# expansiva: treina ate a semana N, testa na N+1, e repete avancando.
!python -u -m pricing.demand.ml_forecast --folds 4

## Nível 9. Os gráficos

**O que este nível faz.** Os níveis 6 e 7 gravaram arquivos CSV na pasta `reports/`. A célula lê
dois deles e desenha as duas imagens que servem numa apresentação.

**Onde isso vira código de verdade.** Os CSVs saem do fim da função `run` em
`src/pricing/pipeline.py`.

### Perguntas e respostas

1. O que o primeiro gráfico mostra?

   Uma barra por categoria, com a elasticidade estimada, e uma linha tracejada no `-1`. Verde
   quer dizer que a categoria passou no filtro do nível 6 e entra no otimizador, cinza quer dizer
   que foi descartada. A barra fina em cima de cada uma é o intervalo de confiança de 95%, ou
   seja, o pedaço da régua onde a resposta verdadeira provavelmente está.

2. Olhando esse gráfico, o que decide se uma categoria pode ser precificada?

   Não é a barra estar depois do `-1`, é a **barra fina inteira** estar. São três: `cool_stuff`,
   `watches_gifts` e `perfumery`.

   Se o intervalo cruza o `-1`, os dois mundos continuam possíveis: o de subir preço aumentar a
   receita e o de derrubar. Mexer no preço dessa categoria é apostar, não decidir.

3. E as duas categorias com valor positivo?

   Ruído. O intervalo delas é tão largo que atravessa o zero. Não são produtos que vendem mais
   quando encarecem, são estimativas sem informação suficiente.

4. O que o segundo gráfico mostra?

   O ganho de margem em cada regime de trava. A barra lima, `+33,34%`, é o ótimo sem trava
   nenhuma, e ela só existe no papel: exige subir 30% o preço médio e perder 28% do volume. A
   primeira verde, `+7,56%`, é a política real. A distância entre as duas, **25,79 pontos**, é o
   preço da prudência.

5. Qual dos dois você mostraria primeiro numa reunião?

   O segundo. O primeiro explica por que só três categorias entram, que é discussão técnica. O
   segundo mostra dinheiro e a escolha que a área comercial precisa fazer, que é a conversa que
   eles querem ter.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPORTS = Path("/content/price-optimization-olist/reports")

VERDE, LIMA, CINZA, ESCURO = "#197A56", "#96F878", "#C9C4BC", "#0C2B15"

# --- 1. Elasticidade por categoria -------------------------------------
el = pd.read_csv(REPORTS / "elasticity_by_category_olist.csv").sort_values("elasticity")

# Verde = passou na regua e entra no otimizador. Cinza = reprovou.
cores = [VERDE if bool(c) else CINZA for c in el["credible"]]

fig, ax = plt.subplots(figsize=(9, 7))
# xerr desenha o intervalo de confianca de 95%, que e ~1,96 erros padrao.
ax.barh(el["category"], el["elasticity"], color=cores,
        xerr=1.96 * el["std_error"], error_kw={"ecolor": "#8A8580", "lw": 1})
ax.axvline(-1, ls="--", lw=1.2, color=ESCURO)   # a fronteira que decide tudo
ax.axvline(0, lw=0.8, color="#555")
ax.set_xlabel("elasticidade estimada, barra fina = intervalo de 95%")
ax.set_title("Elasticidade por categoria, Olist real\nverde = entra no otimizador")
plt.tight_layout()
plt.show()

aprovadas = el[el["credible"]]
print(f"{len(aprovadas)} de {len(el)} categorias entram no otimizador: "
      + ", ".join(aprovadas["category"]))

# --- 2. O que cada trava custa -----------------------------------------
fr = pd.read_csv(REPORTS / "guardrail_frontier_olist.csv").iloc[::-1]

# Lima = otimo sem trava nenhuma.
cores2 = [LIMA if g.startswith("none") else VERDE for g in fr["guardrails"]]

fig, ax = plt.subplots(figsize=(9, 4))
barras = ax.barh(fr["guardrails"], fr["margin_uplift_pct"], color=cores2)
ax.bar_label(barras, fmt="%+.2f%%", padding=4, fontsize=9)
ax.axvline(0, lw=0.8, color="#555")
ax.set_xlabel("ganho de margem (%)")
ax.set_title("O que cada trava de negocio custa")
ax.set_xlim(right=fr["margin_uplift_pct"].max() * 1.3)
plt.tight_layout()
plt.show()

display(fr.iloc[::-1][["guardrails", "margin_uplift_pct", "volume_change_pct",
                       "avg_price_change_pct", "margin_given_up_pp"]].round(2))

## Nível 10. A suíte de testes

**O que este nível faz.** Roda os testes automatizados do repositório. É o mesmo comando que o
GitHub roda sozinho a cada push, e é ele que dá o selo verde na página do projeto.

**Onde isso vira código de verdade.** Pasta `tests/`, e o `pytest.ini` na raiz, que é quem faz o
comando funcionar sem você configurar nada.

### Perguntas e respostas

1. Para que serve teste automatizado num projeto de análise, se não tem usuário nenhum usando
   isso em produção?

   Para te impedir de acreditar no seu próprio resultado cedo demais. Num sistema comum o teste
   evita que o usuário veja um erro. Aqui ele evita que **você apresente um número errado**
   achando que está certo.

2. Isso já aconteceu neste projeto?

   Sim, e vale contar numa entrevista. Um teste pegou um bug antes do commit: a média móvel das
   4 últimas semanas, que é uma das 9 informações do nível 8, estava escorregando de um produto
   para o outro. As primeiras semanas de um produto entravam no histórico do produto anterior.

   O efeito era o pior possível: o resultado ficava **melhor** do que devia, porque o modelo
   recebia informação que não era dele. Um bug que piora o número você percebe. Um que melhora,
   você comemora.

3. Como um teste consegue conferir um resultado estatístico, e não só o retorno de uma função?

   Pelo mesmo truque do nível 4: ele gera um dado pequeno com a resposta já conhecida e checa se
   o estimador chega perto dela. Não dá para testar contra a verdade do Olist, porque ela não
   existe, mas dá para testar contra uma verdade que você mesmo plantou.

In [ ]:
# Alvo "test" do Makefile. Roda sem configuracao nenhuma porque o pytest.ini
# na raiz ja declara "pythonpath = src" e "testpaths = tests".
# E o mesmo comando que o GitHub roda a cada push.
!python -m pytest -q

## Fecho

Se você chegou aqui, consegue contar a história inteira sem slide:

1. Qual é a pergunta de negócio, e por que ela não se responde com uma média.

   A pergunta é: que preço colocar em cada produto para ganhar mais margem sem derrubar o
   volume. Média não responde porque os preços do histórico não foram sorteados. Produto bom
   é caro e vende muito, produto ruim é barato e vende pouco, então a média mostra preço alto
   andando junto com venda alta e sugere que subir preço aumenta a venda, que é o contrário
   da verdade.

2. Por que estimar elasticidade em dado observacional é difícil, e o que é endogeneidade.

   Endogeneidade é preço e venda se mexerem juntos por causa de uma terceira coisa que você
   não observa, aqui a qualidade do produto. O preço não foi definido por sorteio, foi
   definido por gente olhando exatamente o que falta na sua tabela. Por isso a correlação
   entre preço e venda mistura duas histórias e você não sabe qual está vendo.

3. Como você sabe que o método funciona, já que a resposta certa não existe no dado real.

   Pelo nível 4. Num mercado inventado, com gabarito de `-2,103` (cada 1% de aumento no preço
   derruba 2,103% da venda), o 2SLS devolveu `-2,103` e o intervalo de confiança dele conteve
   a verdade em 20 das 20 rodadas. O OLS errou 24% para o lado perigoso e o intervalo dele não
   conteve a verdade nenhuma vez.

4. O que aconteceu quando o método encostou no dado de verdade.

   O instrumento morreu. F do primeiro estágio de `0,16` contra um mínimo de 10, porque apenas
   1,1% da variação de preço acontece dentro do mesmo produto. O pipeline recusou o próprio
   2SLS e caiu para efeitos fixos, `-0,756`. E só 3 categorias de 20 puderam ser precificadas,
   o que corresponde a 23% da receita do painel.

5. Por que existe um otimizador, e não uma fórmula.

   A fórmula existe, é `p* = c * beta / (1 + beta)`, e está no próprio repositório. Com custo
   R\$ 26,22 e beta `-2,103` ela dá `26,22 x (-2,103) / (-1,103)` = **R\$ 50**. Só que ela
   precisa de demanda elástica para ter solução: com o `-0,756` da carteira o `(1 + beta)` fica
   positivo, a divisão troca de sinal e a conta devolve preço negativo, o que na prática quer
   dizer "cobre mais para sempre". E mesmo onde ela funciona, ela decide um produto de cada
   vez, ignorando que a empresa tem limite de preço médio, de receita e de volume no total. Com
   esses limites, os produtos precisam trocar folga entre si, e isso é um problema de
   otimização, não uma conta fechada.

6. Qual é a maior fraqueza do trabalho, e o que resolveria.

   O Olist não registra custo. A margem de 35% é premissa, não dado, e o ganho de `+7,56%`
   depende dela: vai de `+17,5%` com margem de 20% a `+2,2%` com margem de 60%. A direção é
   robusta, o tamanho não. O que resolveria é custo de verdade, ou um teste A/B alternando
   preços no tempo, que mediria o efeito sem depender de modelo nenhum.

A sexta é a que separa quem apresentou de quem entendeu, e é a única que ninguém espera que
você levante sozinho. Dizer ela antes de perguntarem vale mais do que ser pego.